# FlashInspector AI – Test Model on Video

Upload a video and run your trained YOLOv8 models on it.

Runs **both** models on every frame and combines the annotations:
- **Detection** (`best_detect.pt`) — bounding boxes for fire safety equipment (S1,2,5,6)
- **Segmentation** (`best_segment.pt`) — masks for extinguisher condition/occlusion (S3,4)

**Steps:** Check GPU → Install deps → Get code → Upload models → Upload video → Run inference → View & download results

## 1. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** if no GPU is detected.

In [ ]:
!nvidia-smi 2>/dev/null || echo "\n\u26a0\ufe0f No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU."

## 2. Install dependencies & clone repo

In [ ]:
!pip install -q ultralytics opencv-python easyocr
!git clone --depth 1 https://github.com/patrisiyarum/fire.git /content/fire 2>/dev/null || echo "Repo already cloned."
%cd /content/fire/flashinspector-ai

from pathlib import Path
print("\nProject ready at:", Path.cwd())
print("Model exists:", Path("best.pt").exists())

## 3. Upload your models

Upload both models from Kaggle:
- **`best_detect.pt`** — detection (fire blankets, extinguishers, smoke detectors, marking signs, emergency exits, inspection tags, fire class symbols, etc.)
- **`best_segment.pt`** — segmentation (fire extinguisher condition/occlusion)

Upload both files at once, or upload just one — the notebook will use whichever models are available.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

# Upload your models (select both best_detect.pt and best_segment.pt at once)
print("Upload your model files (best_detect.pt and/or best_segment.pt):")
print("You can select multiple files in the upload dialog.\n")

detect_model = None
segment_model = None

try:
    up = files.upload()
    uploaded_names = list(up.keys())
except Exception:
    uploaded_names = []

# Load uploaded models
for name in uploaded_names:
    p = Path(name)
    m = YOLO(str(p))
    t = m.overrides.get("task", "detect")
    if t == "segment":
        segment_model = m
        print(f"Segmentation model loaded: {p}")
        print(f"  Classes: {m.names}\n")
    else:
        detect_model = m
        print(f"Detection model loaded: {p}")
        print(f"  Classes: {m.names}\n")

# Fallback to repo best.pt if nothing uploaded
if detect_model is None and segment_model is None:
    fallback = Path("best.pt")
    if fallback.exists():
        detect_model = YOLO(str(fallback))
        print(f"Using fallback model from repo: {fallback}")
        print(f"  Classes: {detect_model.names}\n")
    else:
        raise FileNotFoundError("No models found. Upload best_detect.pt and/or best_segment.pt.")

# Summary
models_loaded = []
if detect_model:
    models_loaded.append("Detection")
if segment_model:
    models_loaded.append("Segmentation")
print(f"Models ready: {', '.join(models_loaded)}")

## 4. Upload your video

In [ ]:
from google.colab import files
from pathlib import Path

print("Select a video file (.mp4, .avi, .mov, .mkv):")
uploaded = files.upload()
VIDEO_PATH = Path(list(uploaded.keys())[0])
print(f"\nUploaded: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 5. Run inference on video (both models)

Runs both detection and segmentation models on each frame and combines the annotations.
- **CONFIDENCE**: minimum detection confidence (0.0–1.0)
- **FRAME_SKIP**: process every Nth frame (higher = faster, lower = more detailed)

Features:
- **Class consolidation**: "Right Exit", "Left Exit", "Emergency Exit" all shown as `emergency_exit`
- **No duplicate boxes**: segmentation model draws masks only; detection model draws bounding boxes
- **Cross-model NMS**: removes duplicate detections between models
- **OCR label reading**: reads text on detected fire extinguishers (shown in yellow below the box)

In [ ]:
import cv2
import numpy as np
import time
from pathlib import Path

# Settings
CONFIDENCE = 0.25
FRAME_SKIP = 5  # process every 5th frame

# --- Class name consolidation ---
# Maps granular/variant class names to unified display names
CLASS_MAP = {
    "right exit": "emergency_exit",
    "left exit": "emergency_exit",
    "Right Exit": "emergency_exit",
    "Left Exit": "emergency_exit",
    "emergency exit": "emergency_exit",
    "Emergency Exit": "emergency_exit",
    "fire-extinguisher": "fire_extinguisher",
    "fire extinguisher": "fire_extinguisher",
    "Fire_Extinguisher": "fire_extinguisher",
}

def consolidate_class(name):
    """Map variant class names to a single canonical name."""
    return CLASS_MAP.get(name, name)

# --- OCR setup for reading fire extinguisher labels ---
import easyocr
ocr_reader = easyocr.Reader(["en"], gpu=True, verbose=False)
print("OCR reader loaded for fire extinguisher label reading.\n")

def read_label(frame, bbox, padding=10):
    """Crop the detected fire extinguisher region and run OCR on it."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    # Add padding and clamp to frame bounds
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return []
    results = ocr_reader.readtext(crop, detail=0, paragraph=True)
    return results

# --- NMS helper to remove duplicate detections across models ---
def cross_model_nms(det_boxes, seg_boxes, iou_thresh=0.5):
    """Remove segmentation boxes that overlap heavily with detection boxes.
    Returns indices of seg_boxes to KEEP."""
    if not det_boxes or not seg_boxes:
        return list(range(len(seg_boxes)))

    keep = []
    for si, sb in enumerate(seg_boxes):
        overlap = False
        for db in det_boxes:
            iou = _compute_iou(sb, db)
            if iou > iou_thresh:
                overlap = True
                break
        if not overlap:
            keep.append(si)
    return keep

def _compute_iou(box1, box2):
    """Compute IoU between two [x1,y1,x2,y2] boxes."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# Open video
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH.name}")
print(f"Resolution: {width}x{height}, FPS: {fps:.1f}, Total frames: {total_frames}")
print(f"Duration: {total_frames / fps:.1f}s")
active = []
if detect_model:
    active.append("Detection")
if segment_model:
    active.append("Segmentation")
print(f"Running: {' + '.join(active)}")
print(f"Processing every {FRAME_SKIP} frame(s)...\n")

# Output video
out_dir = Path("inference_results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{VIDEO_PATH.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path), fourcc, fps / FRAME_SKIP, (width, height))

# Process frames
detect_detections = []
segment_detections = []
ocr_results_all = []
frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        annotated = frame.copy()
        det_bboxes = []   # for cross-model NMS
        seg_bboxes = []

        # --- Run segmentation model (masks only, no duplicate boxes) ---
        if segment_model:
            seg_results = segment_model(frame, conf=CONFIDENCE, verbose=False)[0]
            # Draw ONLY masks, not bounding boxes (avoids duplicate box annotations)
            annotated = seg_results.plot(img=annotated, boxes=False)
            for box in seg_results.boxes:
                bbox = [float(x) for x in box.xyxy[0].tolist()]
                seg_bboxes.append(bbox)

        # --- Run detection model ---
        if detect_model:
            det_results = detect_model(frame, conf=CONFIDENCE, verbose=False)[0]
            for box in det_results.boxes:
                bbox = [float(x) for x in box.xyxy[0].tolist()]
                det_bboxes.append(bbox)

            # Draw detection boxes with consolidated class names
            for box in det_results.boxes:
                bbox = [int(x) for x in box.xyxy[0].tolist()]
                raw_cls = det_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                conf_val = float(box.conf)

                # Draw bounding box
                color = (0, 255, 0) if "extinguisher" in cls_name else (255, 165, 0)
                cv2.rectangle(annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                label = f"{cls_name} {conf_val:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (bbox[0], bbox[1] - th - 6), (bbox[0] + tw, bbox[1]), color, -1)
                cv2.putText(annotated, label, (bbox[0], bbox[1] - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

                detect_detections.append({
                    "frame": frame_idx,
                    "timestamp_sec": round(frame_idx / fps, 2),
                    "class": cls_name,
                    "confidence": round(conf_val, 3),
                    "model": "detect",
                    "bbox": [float(x) for x in box.xyxy[0].tolist()],
                })

                # --- OCR: read label text on fire extinguishers ---
                if "extinguisher" in cls_name.lower():
                    label_texts = read_label(frame, box.xyxy[0].tolist())
                    if label_texts:
                        ocr_results_all.append({
                            "frame": frame_idx,
                            "timestamp_sec": round(frame_idx / fps, 2),
                            "texts": label_texts,
                        })
                        # Draw OCR text below the bounding box
                        y_offset = bbox[3] + 18
                        for txt in label_texts[:3]:  # show up to 3 lines
                            cv2.putText(annotated, txt[:60], (bbox[0], y_offset),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)
                            y_offset += 18

        # --- Record segmentation detections (only non-duplicate ones) ---
        if segment_model and seg_results.boxes:
            keep_idx = cross_model_nms(det_bboxes, seg_bboxes, iou_thresh=0.5)
            for i, box in enumerate(seg_results.boxes):
                if i in keep_idx:
                    segment_detections.append({
                        "frame": frame_idx,
                        "timestamp_sec": round(frame_idx / fps, 2),
                        "class": consolidate_class(seg_results.names[int(box.cls)]),
                        "confidence": round(float(box.conf), 3),
                        "model": "segment",
                    })

        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% done — {processed} frames processed ({processed / elapsed:.1f} fps)")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
all_detections = detect_detections + segment_detections
print(f"\nDone! Processed {processed} frames in {elapsed:.1f}s ({processed / max(elapsed, 0.01):.1f} fps)")
print(f"Annotated video saved: {out_path}")
if ocr_results_all:
    print(f"OCR: Read text from fire extinguishers in {len(ocr_results_all)} frames")

## 6. Results summary

Shows detection and segmentation results separately, then combined totals.

In [ ]:
from collections import Counter

def print_summary(title, detections):
    if not detections:
        print(f"\n{title}: No detections.")
        return
    counts = Counter(d["class"] for d in detections)
    print(f"\n{'='*50}")
    print(f"{title}")
    print(f"{'='*50}")
    print(f"Total: {len(detections)} detections\n")
    for cls, count in counts.most_common():
        confs = [d["confidence"] for d in detections if d["class"] == cls]
        print(f"  {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")

# Detection model results
if detect_model:
    print_summary("DETECTION MODEL (S1,2,5,6)", detect_detections)

# Segmentation model results
if segment_model:
    print_summary("SEGMENTATION MODEL (S3,4)", segment_detections)
    seg_frames = len(set(d["frame"] for d in segment_detections))
    print(f"\n  Masks drawn on {seg_frames} frames")

# Combined
if detect_detections or segment_detections:
    print(f"\n{'='*50}")
    print(f"COMBINED: {len(all_detections)} total detections")
    print(f"{'='*50}")
    all_counts = Counter(d["class"] for d in all_detections)
    for cls, count in all_counts.most_common():
        confs = [d["confidence"] for d in all_detections if d["class"] == cls]
        src = "detect" if any(d["class"] == cls and d["model"] == "detect" for d in all_detections) else "segment"
        print(f"  {cls}: {count} [{src}] (avg conf: {sum(confs)/len(confs):.2f})")

    print(f"\nFirst 10 detections:")
    for d in all_detections[:10]:
        print(f"  [{d['timestamp_sec']}s] [{d['model']}] {d['class']}: {d['confidence']}")
else:
    print("No detections found in the video.")

# OCR results (fire extinguisher label readings)
if ocr_results_all:
    print(f"\n{'='*50}")
    print(f"FIRE EXTINGUISHER LABEL READINGS (OCR)")
    print(f"{'='*50}")
    print(f"Labels read from {len(ocr_results_all)} frames:\n")
    # Deduplicate OCR text across frames
    seen_texts = set()
    unique_readings = []
    for ocr in ocr_results_all:
        for txt in ocr["texts"]:
            normalized = txt.strip().lower()
            if normalized and normalized not in seen_texts:
                seen_texts.add(normalized)
                unique_readings.append({"text": txt.strip(), "timestamp": ocr["timestamp_sec"]})
    for r in unique_readings[:20]:
        print(f"  [{r['timestamp']}s] \"{r['text']}\"")
    if len(unique_readings) > 20:
        print(f"  ... and {len(unique_readings) - 20} more unique readings")
else:
    print("\nNo fire extinguisher label text detected by OCR.")

## 7. Preview a sample frame

In [ ]:
import cv2
from IPython.display import display, Image as IPImage
from pathlib import Path
import tempfile

# Open the annotated video and grab a frame from the middle
cap = cv2.VideoCapture(str(out_path))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # jump to middle
ret, frame = cap.read()
cap.release()

if ret:
    tmp = Path(tempfile.mktemp(suffix=".jpg"))
    cv2.imwrite(str(tmp), frame)
    display(IPImage(str(tmp), width=800))
    print(f"Showing frame {total // 2} of {total}")
else:
    print("Could not read a frame from the output video.")

## 8. Download the annotated video

In [ ]:
from google.colab import files
from pathlib import Path

if Path(out_path).exists():
    print(f"Downloading: {out_path} ({Path(out_path).stat().st_size / 1024 / 1024:.1f} MB)")
    files.download(str(out_path))
else:
    print("Output video not found. Run the inference cell above first.")